In [ ]:
import os
from snowflake.snowpark import Session, Window
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import (StructType, StructField, IntegerType,
                                      StringType, DateType, DecimalType)
from practice_data import TABLES, typed_rows, EXPECTED_COUNTS

session = Session.builder.configs({
    "account":   os.environ["SNOWFLAKE_ACCOUNT"],
    "user":      os.environ["SNOWFLAKE_USER"],
    "password":  os.environ["SNOWFLAKE_PASSWORD"],
    "role":      "SYSADMIN",
    "warehouse": "PRACTICE_WH",
    "database":  "SQL_PRACTICE",
    "schema":    "PATTERNS",
}).create()

SP_TYPES = {"INT": IntegerType(), "STRING": StringType(), "DATE": DateType()}

def sp_type(kind):
    return DecimalType(10, 2) if kind.startswith("DECIMAL") else SP_TYPES[kind]

def load_snowpark(name):
    schema = StructType([StructField(c, sp_type(k))
                         for c, k in TABLES[name]["types"].items()])
    (session.create_dataframe(typed_rows(name), schema=schema)
            .write.mode("overwrite").save_as_table(name))
    return session.table(name)

customers = load_snowpark("customers_dedup")
scores    = load_snowpark("scores")
orders    = load_snowpark("rt_orders")
items     = load_snowpark("rt_order_items")

# sanity check
for name, df in [("customers_dedup", customers), ("scores", scores),
                 ("rt_orders", orders), ("rt_order_items", items)]:
    assert df.count() == EXPECTED_COUNTS[name], name
print("Snowpark data loaded")